# Timme et al. (2016): the model claims (chain model, Bethe lattice, cortical branching model)

Paper: Timme et al. (2016), Front. Physiol. 7:425, doi:10.3389/fphys.2016.00425; claims file
`claims/crcns-hc-8/10.3389-fphys.2016.00425.yaml` (version of record, PMC5037237, with the
supplement). These claims are simulations and use no recorded data; the dataset, CRCNS hc-8
(doi:10.6080/K0PC308P, version `unversioned`, see `culture_criticality.ipynb`), is not read here.
The models port `bethelattice.m` and `cbmodel.m` of the NCC toolbox (see `models.py`).

| claim | here |
|---|---|
| fig3-chain-model-complexity | tested |
| fig5d-bethe-small-avalanches | tested |
| fig5e-shuffle-removes-complexity | tested |
| fig5f-network-state-poisson-keeps-complexity | tested |
| fig6-cb-distribution-curvature | tested |
| fig7c-cb-curvature-no-peak | tested on the 0.01 grid (reading b) |
| fig7ab-cb-complexity-susceptibility-peak | tested on its susceptibility half, 0.01 grid (reading b); the complexity of 30 sub-samples per model was not computed (too large) |
| fig7a-complexity-boundary-rise | blocked, too_large (needs the complexity of fig7ab) |
| figs2-recording-length | blocked, too_large |
| fig7d-cb-branching-ratio-near-1 | tested against the peaks found in fig7ab |


In [ ]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(v, "1")
import time, resource
import multiprocessing as mp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import hc8, models

SEED = 20162
QUICK = bool(os.environ.get("MODELS_QUICK"))     # only while developing
N_SUBSETS = 100     # random subsets per size for complexity (Methods 2.9)
N_DRAWS = 5         # repeated draws of the complexity estimate for the Bethe claims (spread only)
BETHE_RUNS = 10 ** 4        # claims file: a standard choice
CB_MODELS = 30 if not QUICK else 2               # Methods 2.4
CB_STEPS = 3 * 10 ** 5 if not QUICK else 3 * 10 ** 4   # Methods 2.4
GRID = np.round(np.arange(0.20, 0.30 + 1e-9, 0.01), 2)  # claims file, p_trans grid reading (b)
t0 = time.time()
with mp.get_context("spawn").Pool(4) as pool:
    cb_parts = pool.map(models.cb_point, [(float(p), CB_MODELS, CB_STEPS, [SEED, j]) for j, p in enumerate(GRID)])
cb = pd.DataFrame([r for part in cb_parts for r in part])
print(f"cortical branching models: {len(cb)} ({CB_MODELS} per p_trans, {len(GRID)} values, {CB_STEPS} steps) in "
      f"{(time.time() - t0) / 60:.1f} min")

def draws(X, key):
    """The complexity estimate repeated N_DRAWS times with other generators (subsets and correction)."""
    return pd.DataFrame([hc8.complexity(X, N_SUBSETS, np.random.default_rng([SEED, key, r]), return_variants=True)
                         for r in range(N_DRAWS)])


def interior_maxima(m):
    """Grid values whose mean is above both neighbours."""
    v = m.values
    return [float(m.index[i]) for i in range(1, len(v) - 1) if v[i] > v[i - 1] and v[i] > v[i + 1]]

arg_min = lambda s: s.dropna().idxmin() if s.notna().any() else None
arg_max = lambda s: s.dropna().idxmax() if s.notna().any() else None

In [ ]:
print("chain model: 12 neurons, c in {0, 0.8, 1}, exact joint distribution")
print(f"Bethe lattice: {BETHE_RUNS} runs, p_trans 0.5, stop layer 10^4; small avalanches end by layer 8 (255 neurons)")
print(f"cortical branching model: 100 neurons (10 x 10 torus), p_spont 1e-4, {CB_STEPS} steps, {CB_MODELS} models per p_trans")
g = cb.groupby("p_trans")
print(pd.DataFrame({"avalanches (median)": g.n_av.median(), "active fraction (median)": g.active_frac.median().round(3)}).T.to_string())

## fig3-chain-model-complexity

> In the 12-neuron chain model, random data (c = 0) give zero integration, ordered data (c = 1)
> give high integration, and only the complex data (c = 0.8) give non-zero complexity.

In [ ]:
for c in (0.0, 0.8, 1.0):
    curve = models.chain_integration_curve(c)
    print(f"c = {c}: whole-system integration {curve[-1]:.6f} bits; integration at k = 2, 6, 12: "
          f"{np.round(curve[[1, 5, 11]], 4)}; complexity {models.complexity_from_curve(curve):.8f} bits")

**Result: reproduced.** The whole-system integration is 0 bits for c = 0 and 11 bits for c = 1 (5.84 for c = 0.8); the complexity is 0 for c = 0 and c = 1 to eight decimals and 0.431 bits for c = 0.8.

## fig5d-bethe-small-avalanches

> In the critical Bethe lattice model (p_trans = 0.5), avalanche sizes are power-law distributed and
> show shape collapse, and the small avalanches (stopped by the 8th layer) are power-law distributed
> in size and have non-zero complexity.

In [ ]:
rng5 = np.random.default_rng([SEED, 5])
fs_full, fd_full, fprof = models.bethe_full_sizes(BETHE_RUNS, rng5)
fit_full = hc8.plparams(hc8.apply_cuts(fs_full), rng5)
print(f"full model: {BETHE_RUNS} runs, largest size {fs_full.max()}, longest {fd_full.max()} layers; size fit after cuts: tau {fit_full['tau']}, "
      f"range {fit_full['xmin']} to {fit_full['xmax']}, p {fit_full['p']}")
keep = [p for p, d in zip(fprof, fd_full) if d >= 4]
u, c = np.unique(fd_full[fd_full >= 4], return_counts=True)
prof_full = [np.mean([p for p, d in zip(fprof, fd_full) if d == T], axis=0) for T in u[c >= 20]]
print("full model shape collapse (durations >= 4 seen >= 20 times): 1/sigma nu z %.3f, curvature %.3f" % hc8.shape_collapse(prof_full))
raster5, nb5, sizes5 = models.bethe_small_raster(BETHE_RUNS, rng5)
act5 = hc8.activity(raster5, nb5)
s5, d5, st5 = hc8.avalanches(act5)
print(f"small avalanches: {len(sizes5)} of {BETHE_RUNS} runs; sizes 1 to {sizes5.max()}; avalanches in the raster {s5.size} "
      f"(sizes match the runs: {np.array_equal(np.sort(s5), np.sort(sizes5))})")
fit5 = hc8.plparams(hc8.apply_cuts(s5), rng5)
fit5_nocut = hc8.plparams(s5, rng5)
print(f"size fit after the cuts of Methods 2.7: tau {fit5['tau']}, range {fit5['xmin']} to {fit5['xmax']}, p {fit5['p']} "
      f"(accepted: {fit5['p'] >= 0.2 if np.isfinite(fit5['p']) else False}); without cuts: tau {fit5_nocut['tau']}, "
      f"range {fit5_nocut['xmin']} to {fit5_nocut['xmax']}, p {fit5_nocut['p']}")
bins5 = np.flatnonzero(act5) + 1
X5 = hc8.raster_matrix(raster5, bins5)
keep5 = np.asarray(X5.sum(axis=0)).ravel() > 0
cx5 = hc8.complexity(X5[:, keep5], N_SUBSETS, np.random.default_rng([SEED, 51]), return_variants=True)
print(f"complexity (correction and cut, {N_SUBSETS} subsets per size): {cx5['cut']:.5f} bits/neuron over {cx5['N']} neurons "
      f"({cx5['k_kept']} sizes kept); variants: no cut {cx5['uncut']:.5f}, no correction {cx5['raw']:.5f}")
d5 = draws(X5[:, keep5], 51)
print(f"repeated draws ({N_DRAWS}): cut complexity {d5['cut'].min():.5f} to {d5['cut'].max():.5f} (mean {d5['cut'].mean():.5f}), "
      f"sizes kept {d5.k_kept.min()} to {d5.k_kept.max()} of {cx5['N']}; no cut {d5.uncut.min():.4f} to {d5.uncut.max():.4f}; "
      f"above zero in {int((d5['cut'] > 0).sum())} of {N_DRAWS}")

**Result: reproduced.** 7018 of 10000 runs end by layer 8. Their sizes after the cuts are accepted as a truncated power law (tau 1.593, sizes 4 to 12, p 0.338), and their complexity is 0.00637 bits/neuron. The full model's sizes are also accepted (tau 1.402, p 0.368).

## fig5e-shuffle-removes-complexity

> In the Bethe lattice small avalanches, neuron shuffling preserves the avalanche size distribution
> but removes the complexity.

In [ ]:
rsh = hc8.rand_shuffle(raster5, nb5, np.random.default_rng([SEED, 52]))
ash = hc8.activity(rsh, nb5)
print("A(t) unchanged by shuffling:", bool(np.array_equal(ash, act5)), "| size distribution equal:",
      bool(np.array_equal(np.sort(hc8.avalanches(ash)[0]), np.sort(s5))))
Xs = hc8.raster_matrix(rsh, np.flatnonzero(ash) + 1)
ks = np.asarray(Xs.sum(axis=0)).ravel() > 0
cxs = hc8.complexity(Xs[:, ks], N_SUBSETS, np.random.default_rng([SEED, 53]), return_variants=True)
print(f"shuffled complexity {cxs['cut']:.5f} vs original {cx5['cut']:.5f}: ratio {cxs['cut'] / cx5['cut']:.3f} (criterion < 0.10); "
      f"variants: no cut {cxs['uncut']:.5f} vs {cx5['uncut']:.5f}, no correction {cxs['raw']:.5f} vs {cx5['raw']:.5f}")
ds = draws(Xs[:, ks], 53)
print(f"repeated draws: shuffled cut complexity {ds['cut'].min():.5f} to {ds['cut'].max():.5f} (sizes kept {ds.k_kept.min()} to {ds.k_kept.max()}); "
      f"ratio of the means {ds['cut'].mean() / d5['cut'].mean():.3f}; largest shuffled / smallest original {ds['cut'].max() / d5['cut'].min():.3f}")

**Result: reproduced.** Shuffling leaves A(t), and so the size distribution, unchanged, and lowers the complexity from 0.00637 to 0.00002, a ratio of 0.003.

## fig5f-network-state-poisson-keeps-complexity

> In the Bethe lattice small avalanches, randomizing the order of whole network states preserves the
> complexity but changes the avalanche size distribution dramatically.

All time bins (including the silent bin between runs) are put in a random order, so whole network
states move to random times (Methods 2.5, paragraph 4, last sentence).

In [ ]:
perm = np.random.default_rng([SEED, 54]).permutation(nb5)        # new position of each bin
rpo = [np.sort(perm[s - 1] + 1) for s in raster5]
apo = hc8.activity(rpo, nb5)
spo = hc8.avalanches(apo)[0]
Xp = hc8.raster_matrix(rpo, np.flatnonzero(apo) + 1)
kp = np.asarray(Xp.sum(axis=0)).ravel() > 0
cxp = hc8.complexity(Xp[:, kp], N_SUBSETS, np.random.default_rng([SEED, 55]), return_variants=True)
print(f"complexity after permuting network states {cxp['cut']:.5f} vs original {cx5['cut']:.5f}: relative difference "
      f"{abs(cxp['cut'] - cx5['cut']) / cx5['cut']:.3f} (criterion < 0.05); no cut {cxp['uncut']:.5f} vs {cx5['uncut']:.5f} "
      f"({abs(cxp['uncut'] - cx5['uncut']) / cx5['uncut']:.3f}); no correction {cxp['raw']:.5f} vs {cx5['raw']:.5f} "
      f"({abs(cxp['raw'] - cx5['raw']) / cx5['raw']:.3f})")
print(f"avalanches: {spo.size} after vs {s5.size} before; largest size {spo.max()} vs {s5.max()}")
fitp = hc8.plparams(hc8.apply_cuts(spo), np.random.default_rng([SEED, 56]))
sd5 = models.tau_sd(fit5["xmin"], fit5["xmax"], fit5["tau"], fit5["n"], np.random.default_rng([SEED, 57])) if np.isfinite(fit5["tau"]) else np.nan
print(f"size fit after permutation: tau {fitp['tau']}, range {fitp['xmin']} to {fitp['xmax']}, p {fitp['p']}; original tau {fit5['tau']} "
      f"+/- {sd5:.3f} (SD of refits on 500 surrogates)")
changed = (not np.isfinite(fitp["p"])) or fitp["p"] < 0.2 or abs(fitp["tau"] - fit5["tau"]) > sd5
print("size distribution no longer the original power law:", changed)
u0, c0 = np.unique(s5, return_counts=True); u1, c1 = np.unique(spo, return_counts=True)
print("fraction of avalanches of size 1: before %.3f, after %.3f" % (c0[u0 == 1].sum() / s5.size, c1[u1 == 1].sum() / spo.size))
dp = draws(Xp[:, kp], 55)
rel = lambda a, b: abs(a - b) / b
print(f"repeated draws: permuted cut complexity {dp['cut'].min():.5f} to {dp['cut'].max():.5f} (sizes kept {dp.k_kept.min()} to {dp.k_kept.max()}); "
      f"relative difference of the means {rel(dp['cut'].mean(), d5['cut'].mean()):.3f}; no cut {rel(dp.uncut.mean(), d5.uncut.mean()):.3f}; "
      f"no correction {rel(dp.raw.mean(), d5.raw.mean()):.3f}")
print(f"spread of the original's own draws: largest relative difference from their mean {(d5['cut'] - d5['cut'].mean()).abs().max() / d5['cut'].mean():.3f}")

**Result: not reproduced.** Permuting the network states changes the size distribution (tau 2.222 against 1.593 +/- 0.058), but the toolbox's complexity falls from 0.00637 to 0.00001, a relative difference of 0.998. Without the cut the difference is 0.049 and without the correction 0.020, both within 5%: the change comes from where the cut falls.

## fig6-cb-distribution-curvature

> In the cortical branching model, low transmission probabilities give size and duration
> distributions that curve downwards, high ones give distributions that curve upwards, and p_trans
> near 0.25 gives the straightest distributions.

In [ ]:
q = cb.groupby("p_trans")[["quad_size", "quad_dur"]].agg(["mean", "std", "count"])
print(q.round(4).to_string())
for col in ("quad_size", "quad_dur"):
    m = cb.groupby("p_trans")[col].mean()
    print(f"{col}: p 0.2 {m[0.2]:.4f} (negative: {m[0.2] < 0}), p 0.3 {m[0.3]:.4f} (positive: {m[0.3] > 0}), "
          f"p 0.25 {m[0.25]:.4f}; smallest magnitude among 0.2, 0.25, 0.3 at {arg_min(m.loc[[0.2, 0.25, 0.3]].abs())}; "
          f"over the grid at {arg_min(m.abs())}")

**Result: partially reproduced.** For durations the mean quadratic term is -1.23 at p_trans 0.2, +1.39 at 0.3, and smallest in magnitude at 0.25 (-0.020). For sizes it is +0.087 at 0.2, so the low-p_trans size distributions do not curve downwards by this measure, and the smallest magnitude is at 0.21.

## fig7c-cb-curvature-no-peak

> The shape collapse curvature of the cortical branching model does not peak near the critical
> point; sub-critical avalanches are more curved than super-critical ones.

In [ ]:
cv = cb.groupby("p_trans")[["curv_ss", "curv_full"]].agg(["mean", "std"])
print(cv.round(3).to_string())
m = cb.groupby("p_trans").curv_ss.mean()
peaks = interior_maxima(m)
near = [p for p in peaks if abs(p - 0.25) <= 0.02 or abs(p - 0.265) <= 0.02]
print(f"extrapolated |curvature|: p 0.2 {m[0.2]:.3f} vs p 0.3 {m[0.3]:.3f} (0.2 larger: {m[0.2] > m[0.3]}); interior maxima at {peaks}; "
      f"within 0.02 of 0.25 or 0.265: {near}")
mf = cb.groupby("p_trans").curv_full.mean()
print(f"full 100-neuron system: p 0.2 {mf[0.2]:.3f} vs 0.3 {mf[0.3]:.3f}; largest at {arg_max(mf)}, interior maxima {interior_maxima(mf)}")
print("models with a failed shape collapse (NaN):", int(cb.curv_ss.isna().sum()))

**Result: not reproduced.** The extrapolated absolute curvature is larger at p_trans 0.2 (4.258) than at 0.3 (2.999), but it has an interior maximum at 0.25 (4.667), so it does peak near the critical point on the 0.01 grid.

## fig7ab-cb-complexity-susceptibility-peak

> Across cortical branching models, the complexity and the susceptibility (both extrapolated to
> infinite size by sub-sampling) peak near p_trans 0.265, close to the infinite system critical
> point of 0.25.

The criterion needs both peaks within 0.0265 of 0.265. The susceptibility, extrapolated by
sub-sampling (layout reading a), is computed for both readings of Eq. 14 on the 0.01 grid (reading b). The
complexity of 30 sub-samples of each model was not computed: with 100 subsets per size, one
whole-system complexity of a model at p_trans 0.3 took 134 s in development, and the claim needs
9,900 sub-sample computations on this grid and 18,900 on the 0.005 grid.

In [ ]:
print(cb.groupby("p_trans")[["chi_a_ss", "chi_b_ss", "chi_a_full", "chi_b_full"]].mean().to_string(float_format=lambda v: f"{v:.5g}"))
chi_peaks = []
for col, lab in (("chi_a_ss", "reading (a) var(A/N)"), ("chi_b_ss", "reading (b) N var(A/N)")):
    m = cb.groupby("p_trans")[col].mean()
    peaks = interior_maxima(m)
    chi_peaks += peaks
    print(f"extrapolated susceptibility {lab}: largest at p_trans {arg_max(m)}; interior maxima {peaks}; "
          f"within 0.0265 of 0.265: {[p for p in peaks if abs(p - 0.265) <= 0.0265]}; rises at every step: {bool(np.all(np.diff(m.values) > 0))}")
mf = cb.groupby("p_trans").chi_b_full.mean()
print(f"whole 100-neuron system N var(A/N): largest at {arg_max(mf)}, interior maxima {interior_maxima(mf)}")

**Result: ...** (written after the run)

## fig7d-cb-branching-ratio-near-1

> The cortical branching model's branching ratio is closest to 1 near the same transmission
> probability at which complexity and susceptibility peak.

B_new from the whole system (delays 1 to 8, A(t) <= 5), compared with the peaks found under
fig7ab-cb-complexity-susceptibility-peak. For comparison with the model's design (four neighbours,
critical at p_trans 0.25), the mean avalanche size, B_basic (Eq. 11) and the one-step ratio
sum A(t+1) / sum A(t) are printed against 4 p_trans.

In [ ]:
ref = cb.groupby("p_trans")[["B_new", "br_basic", "one_step", "mean_size", "n_av"]].mean()
ref["4 p_trans"] = 4 * ref.index.values
print(ref.to_string(float_format=lambda v: f"{v:.4f}"))
pb = float((ref.B_new - 1).abs().idxmin())
pk = sorted(set(chi_peaks))
print(f"mean B_new closest to 1 at p_trans {pb} ({ref.B_new[pb]:.4f}); susceptibility interior maxima {pk}; "
      f"within 0.0265 of one of them: {[p for p in pk if abs(p - pb) <= 0.0265]}; within 0.0265 of the paper's 0.265: {abs(pb - 0.265) <= 0.0265}")
print(f"run time of this notebook so far: {(time.time() - t0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

**Result: ...** (written after the run)

## Summary

| claim | outcome |
|---|---|
| fig3-chain-model-complexity | reproduced |
| fig5d-bethe-small-avalanches | reproduced |
| fig5e-shuffle-removes-complexity | reproduced |
| fig5f-network-state-poisson-keeps-complexity | not reproduced |
| fig6-cb-distribution-curvature | partially reproduced |
| fig7c-cb-curvature-no-peak | not reproduced |
| fig7ab-cb-complexity-susceptibility-peak | blocked, too_large |
| fig7a-complexity-boundary-rise | blocked, too_large (depends on fig7ab) |
| figs2-recording-length | blocked, too_large |
| fig7d-cb-branching-ratio-near-1 | blocked, too_large (depends on fig7ab) |

Differences from the paper's method: 10 random subsets per size for complexity; the p_trans grid
of 0.01 only (reading b), as 0.005 would double the simulation time; curvature of the
distributions as a quadratic term, a measure the paper does not define. These are simulations, so
they say nothing about the dataset.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.